In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/mirichoi0218/insurance/insurance.csv


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df=pd.read_csv('/kaggle/input/datasets/mirichoi0218/insurance/insurance.csv')

In [4]:
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [5]:
df['region'].value_counts()

region
southeast    364
southwest    325
northwest    325
northeast    324
Name: count, dtype: int64

In [6]:
df.isnull().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

In [7]:
# Sex- OneHotEnocde
# region - OneHotEncode
# Smoker - LabelEncoder

In [8]:
from sklearn.preprocessing import StandardScaler,OneHotEncoder,LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import RidgeCV,LassoCV

In [9]:
x=df.drop('charges',axis=1)
y=df['charges']

In [10]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2)

In [11]:
cat_pipe=Pipeline([
    ('ohe',OneHotEncoder(drop='first'))
    
])

In [12]:
num_pipe=Pipeline([
    ('ss',StandardScaler())
])

In [13]:
preprocess=ColumnTransformer([
    ('cat_pipe',cat_pipe,['sex','smoker','region']),
    ('num_pipe',num_pipe,['age','bmi'])
],remainder='passthrough')

## Linear regression

In [14]:
pipe=Pipeline([
    ('preprocess',preprocess),
    ('lr',LinearRegression())
])

pipe.fit(x_train,y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocess',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat_pipe',
                                                  Pipeline(steps=[('ohe',
                                                                   OneHotEncoder(drop='first'))]),
                                                  ['sex', 'smoker', 'region']),
                                                 ('num_pipe',
                                                  Pipeline(steps=[('ss',
                                                                   StandardScaler())]),
                                                  ['age', 'bmi'])])),
                ('lr', LinearRegression())])

In [15]:
y_pred=pipe.predict(x_test)

from sklearn.metrics import mean_squared_error,r2_score

print('r2 Score :',r2_score(y_test,y_pred))
print('mean_square_error :',mean_squared_error(y_test,y_pred))

r2 Score : 0.7753149722411061
mean_square_error : 34026339.70361335


## Ridge Regression

In [16]:
alphas=np.logspace(-4,4,100)

In [17]:
pipe1=Pipeline([
    ('preprocess',preprocess),
    ('rc',RidgeCV(alphas=alphas,cv=5))
])

pipe1.fit(x_train,y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocess',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat_pipe',
                                                  Pipeline(steps=[('ohe',
                                                                   OneHotEncoder(drop='first'))]),
                                                  ['sex', 'smoker', 'region']),
                                                 ('num_pipe',
                                                  Pipeline(steps=[('ss',
                                                                   StandardScaler())]),
                                                  ['age', 'bmi'])])),
                ('rc',
                 RidgeCV(alphas=array([1.00000000e-04, 1.20450354e-04, 1.45082878e-04, 1.74752840e-04,
       2.10490414e...
       1.38488637e+02, 1.66810054e+02, 2.00923300e+02, 2.42012826e+02,
       2.91505306e+02, 3.51119173e+02, 4.22924287e+02, 5.09413801e+02,
       6.13590727e+02, 7.39072203e+02, 8.90215085e+02, 1.07226722e+03,
       1.29154967e+03, 1.55567614e+03, 1.87381742e+03, 2.25701972e+03,
       2.71858824e+03, 3.27454916e+03, 3.94420606e+03, 4.75081016e+03,
       5.72236766e+03, 6.89261210e+03, 8.30217568e+03, 1.00000000e+04]),
                         cv=5))])

In [18]:
y_pred1=pipe1.predict(x_test)

from sklearn.metrics import mean_squared_error,r2_score

print('r2 Score :',r2_score(y_test,y_pred1))
print('mean_square_error :',mean_squared_error(y_test,y_pred1))
print('Best Alpha :',pipe1['rc'].alpha_)
print('Ridge Coefficient :',pipe1['rc'].coef_)

r2 Score : 0.7750854802792124
mean_square_error : 34061094.00625882
Best Alpha : 0.7564633275546291
Ridge Coefficient : [  236.43816048 23544.98744756  -525.10677842 -1038.24449587
 -1334.63632321  3494.50562164  2102.19260075   598.36642614]


## Lasso Regression

In [19]:
pipe2=Pipeline([
    ('preprocess',preprocess),
    ('lasso',LassoCV(alphas=alphas,cv=5))
])

pipe2.fit(x_train,y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocess',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat_pipe',
                                                  Pipeline(steps=[('ohe',
                                                                   OneHotEncoder(drop='first'))]),
                                                  ['sex', 'smoker', 'region']),
                                                 ('num_pipe',
                                                  Pipeline(steps=[('ss',
                                                                   StandardScaler())]),
                                                  ['age', 'bmi'])])),
                ('lasso',
                 LassoCV(alphas=array([1.00000000e-04, 1.20450354e-04, 1.45082878e-04, 1.74752840e-04,
       2.104904...
       1.38488637e+02, 1.66810054e+02, 2.00923300e+02, 2.42012826e+02,
       2.91505306e+02, 3.51119173e+02, 4.22924287e+02, 5.09413801e+02,
       6.13590727e+02, 7.39072203e+02, 8.90215085e+02, 1.07226722e+03,
       1.29154967e+03, 1.55567614e+03, 1.87381742e+03, 2.25701972e+03,
       2.71858824e+03, 3.27454916e+03, 3.94420606e+03, 4.75081016e+03,
       5.72236766e+03, 6.89261210e+03, 8.30217568e+03, 1.00000000e+04]),
                         cv=5))])

In [20]:
y_pred2=pipe2.predict(x_test)



print('r2 Score :',r2_score(y_test,y_pred2))
print('mean_square_error :',mean_squared_error(y_test,y_pred2))
print('Best Alpha :',pipe2['lasso'].alpha_)
print('Lasso Coefficient :',pipe2['lasso'].coef_)

r2 Score : 0.7755978823475128
mean_square_error : 33983495.747866936
Best Alpha : 5.857020818056673
Lasso Coefficient : [  207.52594596 23616.16410558  -436.04957968  -952.86085983
 -1246.36496565  3493.34214077  2093.78112139   594.39215542]
